# Databricks round trip
The default cells build and verify a local bundle and prepare the Tines contract. Set `LIVE = True` only after configuring your Databricks profile, Volume root and deployed job ID. Live cells upload evidence and launch billable compute.

In [ ]:
from pathlib import Path
import tempfile
import json
root = Path.cwd()
if not (root / 'examples').exists():
    root = root.parent
assert (root / 'examples/parser_samples.json').exists(), 'Run from the repository or notebooks directory'
from timeline_demo.pipeline import Input, run_pipeline, read_timeline
from timeline_demo.core.manifest import verify_bundle
work = tempfile.TemporaryDirectory()
workdir = Path(work.name)
bundle = workdir / 'bundle'
manifest = run_pipeline([
    Input('cloudtrail', root / 'examples/raw/aws/cloudtrail_real_sample.json'),
    Input('entra_signin', root / 'examples/raw/entra/entra_signin_real_sample.jsonl'),
    Input('crowdstrike_detection', root / 'examples/raw/edr/crowdstrike_detection_real_sample.json'),
], bundle, 'notebook-demo')


In [ ]:
from timeline_demo.integrations.databricks import DatabricksClient, tines_request
from timeline_demo.parsers.common import file_hash
LIVE = False
VOLUME_ROOT = '/Volumes/main/incident_timelines/evidence/bundles'
JOB_ID = 1  # Replace with the deployed publish_timeline job ID
remote = VOLUME_ROOT + '/' + manifest['bundle_id']
contract = tines_request(bundle, remote, JOB_ID)
contract

In [ ]:
if LIVE:
    client = DatabricksClient()
    remote = client.upload_bundle(bundle, VOLUME_ROOT)
    run = client.submit(bundle, remote, JOB_ID)
    print(run)
else:
    print('Prepared request; no Databricks call made.')

In [ ]:
if LIVE:
    print(client.status(run['run_id']))
    restored = workdir/'restored'
    client.download_bundle(remote, restored, file_hash(bundle/'audit_manifest.json'))
    assert verify_bundle(restored)['bundle_id'] == manifest['bundle_id']

On Databricks, the deployed job publishes evidence references, events, receipts, quarantine records and a final publication marker. Its second notebook verifies the row count through `published_timeline`. Use `integrations/databricks/normalize_sources.py` for raw exports already in a Volume.

In [ ]:
work.cleanup()